# TP 1 — Fondements : de l’information à l’intelligence artificielle
### Partie I de la conférence · Licence 1 → Master 2

> **TechForge Club · FST — Université de Nouakchott**  
> Conférence **« L’intelligence artificielle : au-delà de la théorie »** · الذكاء الاصطناعي : ما وراء النظرية  
> Animée par **Dr El Benany Med Mahmoud** · samedi 14 novembre 2026 · 12 h 30 · Amphi 5 – FST

**Trois parcours dans le même notebook — choisissez le vôtre :**

| Repère | Public conseillé | Ce que l’on vous demande |
|:---:|---|---|
| 🟢 **Découverte** | Licence 1 – Licence 2 | Exécuter, observer, modifier une valeur, répondre en une phrase. Aucune démonstration exigée. |
| 🟡 **Approfondissement** | Licence 3 | Refaire les calculs à la main, compléter du code, interpréter les résultats. |
| 🔴 **Expert** | Master 1 – Master 2 | Démontrer, implémenter sans bibliothèque, critiquer le protocole, mener un mini-projet. |

Chaque exercice **À faire** porte un repère. Un étudiant de licence peut s’arrêter aux 🟢 d’une section et passer à la suivante ; un étudiant de master traite les trois niveaux. Les cellules `verifier(...)` contrôlent votre réponse sans afficher la solution.

**Objectifs.** Manipuler les notions qui précèdent tout apprentissage automatique : donnée / information / connaissance, bits et entropie de Shannon, algorithme et complexité, calcul matriciel et matériel, règles explicites contre modèles appris, puis un premier modèle génératif écrit à la main.

**Diapositives associées :** Partie I · 01 à 20.

**Prérequis.** 🟢 savoir exécuter une cellule ; 🟡 boucles et fonctions Python ; 🔴 probabilités élémentaires, NumPy.

| Section | Thème | Diapos | 🟢 | 🟡 | 🔴 | Durée |
|---|---|:---:|:---:|:---:|:---:|:---:|
| 1 | Données, traitements, information, connaissance | 02–03 | ✔ | ✔ | ✔ | 15 min |
| 2 | Entropie de Shannon | 04 | ✔ | ✔ | ✔ | 20 min |
| 3 | Bits, quantification, canal bruité | 05 | ✔ | ✔ | ✔ | 20 min |
| 4 | Algorithmes et complexité | 06 | ✔ | ✔ | ✔ | 20 min |
| 5 | Calcul matriciel, CPU/GPU, kernel par blocs | 07–09 | ✔ | ✔ | ✔ | 20 min |
| 6 | IA symbolique, statistique, hybride | 10–14 | ✔ | ✔ | ✔ | 15 min |
| 7 | Apprendre = minimiser ; trois paradigmes | 15–16 | ✔ | ✔ | ✔ | 20 min |
| 8 | Un modèle génératif en 20 lignes | 18–19 | ✔ | ✔ | ✔ | 20 min |
| 9 | Atelier de la conférence, mini-projets, barème | 20 |  | ✔ | ✔ | projet |

In [ ]:
# Installation (à décommenter une seule fois si nécessaire) :
# %pip install numpy pandas matplotlib scikit-learn ipywidgets
# --- Outils communs aux quatre TP de la conférence ---
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
SEED = 42
rng = np.random.default_rng(SEED)
plt.rcParams.update({'figure.figsize': (7, 4), 'axes.grid': True, 'grid.alpha': .3})

def verifier(nom, reponse, attendu, tol=1e-2):
    """Compare votre réponse à la valeur attendue, sans l'afficher."""
    if reponse is None:
        print(f'⏳ {nom} : à compléter (remplacez None par votre réponse)'); return
    ok = np.allclose(np.asarray(reponse, dtype=float), np.asarray(attendu, dtype=float), atol=tol)
    print(f'✅ {nom} : correct' if ok else f'❌ {nom} : à revoir — reprenez le calcul')

def explorer(f, **curseurs):
    """Curseurs interactifs si ipywidgets est installé ; sinon exécution avec les valeurs par défaut.
    Chaque curseur est donné sous la forme nom=(min, max, pas, valeur_par_defaut)."""
    try:
        import ipywidgets as w
        ctl = {k: (w.FloatSlider if any(isinstance(x, float) for x in v) else w.IntSlider)(
                   min=v[0], max=v[1], step=v[2], value=v[3], description=k, continuous_update=False)
               for k, v in curseurs.items()}
        w.interact(f, **ctl)
    except ImportError:
        print('(ipywidgets absent : résultat pour les valeurs par défaut — changez-les dans l’appel pour explorer)')
        f(**{k: v[3] for k, v in curseurs.items()})
import math, time, zlib
from collections import Counter
print('Environnement prêt. Graine =', SEED)

## 1 — Données, traitements, information, connaissance  *(diapos 02–03)*

Un **programme** applique un **traitement** à des **données** pour produire un **résultat**. Une **donnée** est une valeur brute ; une **information** est une donnée interprétée dans un contexte ; une **connaissance** est une règle, un modèle ou un savoir réutilisable.

In [ ]:
# DONNÉES
a, b = 6, 4
# TRAITEMENT
def multiplier(x, y):
    return x * y
# RÉSULTAT
print('Résultat :', multiplier(a, b))

# Donnée → information → connaissance → décision
notes = [12, 16, 14]                      # données
moyenne = sum(notes) / len(notes)         # information
SEUIL = 10                                # connaissance : règle institutionnelle
print(f'Moyenne = {moyenne:.2f}/20 →', 'validé' if moyenne >= SEUIL else 'non validé')

**À faire**

- 🟢 Changez `a`, `b`, puis les notes. Quelle ligne est une *donnée* ? laquelle est une *connaissance* ?
- 🟡 Écrivez une fonction `decision(notes, seuil)` qui renvoie `'validé'` ou `'non validé'`, puis testez-la avec trois listes de notes.
- 🔴 La règle « moyenne ≥ 10 » ignore les notes éliminatoires. Proposez une règle plus riche (par exemple : aucune note < 6) et discutez : est-ce une donnée, une information ou une connaissance supplémentaire ?

In [ ]:
def decision(notes, seuil=10):
    # TODO 🟡 : calculer la moyenne et renvoyer 'validé' ou 'non validé'
    return None

print(decision([12, 16, 14]), decision([4, 9, 11]))

## 2 — Entropie de Shannon  *(diapo 04)*

L’information apportée par un événement de probabilité $p$ est $I=-\log_2 p$ bits : plus l’événement est rare, plus il informe. L’**entropie** est l’information moyenne d’une source :

$$H(X)=-\sum_i p_i\log_2 p_i .$$

**Calcul à la main.** Pièce équilibrée : $H=-2\times\tfrac12\log_2\tfrac12=1$ bit. Pièce toujours « pile » : $H=0$.

> L’entropie mesure une **incertitude**, pas le sens ni la vérité d’un message.

In [ ]:
def entropie(probas):
    p = np.asarray(probas, dtype=float)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

print('Pièce équilibrée :', entropie([.5, .5]), 'bit')
print('Pièce biaisée 90/10 :', round(entropie([.9, .1]), 3), 'bit')
print('Dé à 6 faces :', round(entropie([1/6] * 6), 3), 'bits')

ps = np.linspace(.001, .999, 300)
plt.plot(ps, [entropie([p, 1 - p]) for p in ps])
plt.xlabel('p(pile)'); plt.ylabel('H (bits)'); plt.title('Entropie d’une pièce'); plt.show()

In [ ]:
def voir_piece(p=0.5):
    print(f'p = {p:.2f} | I(pile) = {-math.log2(p):.3f} bit | I(face) = {-math.log2(1 - p):.3f} bit | H = {entropie([p, 1 - p]):.3f} bit')

explorer(voir_piece, p=(0.01, 0.99, 0.01, 0.5))

### Entropie d’un texte

On estime la probabilité de chaque caractère par sa fréquence. Si les caractères étaient tirés indépendamment, il faudrait en moyenne $H$ bits par caractère, contre 8 bits en codage ASCII.

In [ ]:
texte = ("l'intelligence artificielle regroupe des methodes permettant a des systemes informatiques "
         "d'accomplir des taches associees a des capacites cognitives : percevoir, raisonner, apprendre, "
         "planifier ou generer. une performance elevee sur une tache ne demontre pas une intelligence generale.")
freq = Counter(texte)
probas = np.array(list(freq.values())) / len(texte)
H = entropie(probas)
print(f'{len(texte)} caractères, {len(freq)} symboles différents')
print(f'Entropie ≈ {H:.3f} bits/caractère (contre 8 en ASCII) → gain théorique {100 * (1 - H / 8):.0f} %')
print('Symboles les plus fréquents :', freq.most_common(5))

**À faire**

- 🟢 Dans l’explorateur, pour quelle valeur de $p$ l’entropie est-elle maximale ? Pourquoi ?
- 🟡 Calculez à la main l’entropie d’une source à trois symboles de probabilités $(\tfrac12,\tfrac14,\tfrac14)$, puis vérifiez.
- 🔴 Comparez $H$ au taux obtenu par `zlib` sur le texte répété 50 fois. Pourquoi un compresseur peut-il faire **mieux** que l’entropie d’ordre 0 ? (indice : les caractères ne sont pas indépendants.)

In [ ]:
ma_reponse_entropie = None          # 🟡 votre valeur, en bits
verifier('Entropie de (1/2, 1/4, 1/4)', ma_reponse_entropie, entropie([.5, .25, .25]))

# 🔴 point de départ
long_texte = (texte * 50).encode()
bits_par_car = 8 * len(zlib.compress(long_texte, 9)) / len(long_texte)
print(f'zlib sur le texte répété : {bits_par_car:.3f} bits/caractère — à comparer à H = {H:.3f}')

## 3 — Bits, quantification et canal bruité  *(diapo 05)*

**Taille d’une image.** $100\times100$ pixels à 8 bits/pixel : $100\times100\times8=80\,000$ bits avant compression.

**Quantifier**, c’est réduire le nombre de bits par pixel : $b$ bits donnent $2^b$ niveaux de gris.

In [ ]:
# Image synthétique : un disque clair sur un dégradé
yy, xx = np.mgrid[0:64, 0:64]
image = np.clip(.25 + .5 * xx / 63 + .45 * (np.hypot(xx - 32, yy - 32) < 16), 0, 1)

fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for ax, b in zip(axes, [8, 3, 2, 1]):
    niveaux = 2 ** b
    q = np.round(image * (niveaux - 1)) / (niveaux - 1)
    ax.imshow(q, cmap='gray', vmin=0, vmax=1); ax.axis('off')
    ax.set_title(f'{b} bit(s)/pixel\n{64 * 64 * b:,} bits'.replace(',', ' '))
plt.tight_layout(); plt.show()
print('Image 100 × 100 à 8 bits/pixel :', 100 * 100 * 8, 'bits')

### Transmettre sur un canal bruité : le code à répétition

Chaque bit est inversé avec la probabilité $p$. **Codage correcteur** le plus simple : envoyer chaque bit trois fois et décider à la majorité. Une erreur subsiste si au moins deux copies sur trois sont inversées :

$$P_{\text{erreur}}=3p^2(1-p)+p^3=3p^2-2p^3 .$$

Le prix à payer : trois fois plus de bits transmis.

In [ ]:
def canal(bits, p, rng):
    return bits ^ (rng.random(bits.shape) < p)

def experience_canal(p=0.1, n=20000):
    g = np.random.default_rng(SEED)
    message = g.integers(0, 2, n).astype(bool)
    sans_code = canal(message, p, g)
    recu = canal(np.repeat(message, 3), p, g).reshape(-1, 3)
    decode = recu.sum(axis=1) >= 2
    print(f'p = {p:.2f} | erreurs sans code : {np.mean(sans_code != message):.4f} | '
          f'avec répétition ×3 : {np.mean(decode != message):.4f} | théorie : {3 * p**2 - 2 * p**3:.4f}')

explorer(experience_canal, p=(0.01, 0.5, 0.01, 0.1))

**À faire**

- 🟢 Combien de bits occupe une photo de $1000\times1000$ pixels en couleurs (24 bits/pixel) ? Convertissez en mégaoctets.
- 🟡 Pour $p=0{,}1$, calculez à la main $3p^2-2p^3$ et comparez à la simulation.
- 🔴 Généralisez à un code à répétition ×5 (majorité de 5). Donnez la formule de l’erreur et vérifiez-la par simulation. À partir de quelle valeur de $p$ la répétition n’apporte-t-elle plus rien ?

In [ ]:
ma_taille_mo = None                 # 🟢 taille en mégaoctets (1 Mo = 10**6 octets)
verifier('Photo 1000×1000, 24 bits/pixel', ma_taille_mo, 1000 * 1000 * 24 / 8 / 1e6)
mon_erreur = None                   # 🟡 probabilité d'erreur après décodage pour p = 0,1
verifier('Erreur du code ×3 pour p = 0,1', mon_erreur, 3 * .1**2 - 2 * .1**3, tol=1e-3)

## 4 — Algorithmes et complexité  *(diapo 06)*

Un **algorithme** est une suite finie d’instructions précises ; un **programme** l’implémente. La **complexité** décrit comment le coût croît avec la taille $n$ du problème.

- Chercher le maximum de $n$ valeurs : $n-1$ comparaisons, donc $O(n)$.
- Chercher une valeur dans un tableau **trié** par dichotomie : environ $\log_2 n$ étapes, donc $O(\log n)$.
- Trier par comparaisons successives de voisins (tri à bulles) : $O(n^2)$.

In [ ]:
def maximum(T):
    m, comparaisons = T[0], 0
    for v in T[1:]:
        comparaisons += 1
        if v > m: m = v
    return m, comparaisons

def recherche_lineaire(T, cible):
    for i, v in enumerate(T):
        if v == cible: return i + 1          # nombre d'éléments examinés
    return len(T)

def recherche_dichotomique(T, cible):
    g, d, etapes = 0, len(T) - 1, 0
    while g <= d:
        etapes += 1; m = (g + d) // 2
        if T[m] == cible: return etapes
        if T[m] < cible: g = m + 1
        else: d = m - 1
    return etapes

print('maximum([3, 9, 4, 7]) →', maximum([3, 9, 4, 7]), '(valeur, comparaisons)')
lignes = []
for n in [10, 1_000, 100_000, 1_000_000]:
    T = list(range(n))
    lignes.append({'n': n, 'linéaire (pire cas)': recherche_lineaire(T, n - 1), 'dichotomie': recherche_dichotomique(T, n - 1), 'log2(n)': round(math.log2(n), 1)})
display(pd.DataFrame(lignes))

In [ ]:
def tri_bulles(T):
    T = list(T)
    for i in range(len(T)):
        for j in range(len(T) - 1 - i):
            if T[j] > T[j + 1]: T[j], T[j + 1] = T[j + 1], T[j]
    return T

tailles, t_bulles, t_python = [200, 400, 800, 1600], [], []
for n in tailles:
    T = list(rng.integers(0, 10**6, n))
    t = time.perf_counter(); tri_bulles(T); t_bulles.append(time.perf_counter() - t)
    t = time.perf_counter(); sorted(T); t_python.append(time.perf_counter() - t)
plt.plot(tailles, t_bulles, 'o-', label='tri à bulles — O(n²)')
plt.plot(tailles, t_python, 's-', label='sorted() — O(n log n)')
plt.xlabel('n'); plt.ylabel('temps (s)'); plt.legend(); plt.title('Deux algorithmes corrects, deux coûts'); plt.show()
print('Quand n double, le tri à bulles prend ≈', [round(t_bulles[i + 1] / t_bulles[i], 1) for i in range(3)], 'fois plus de temps')

**À faire**

- 🟢 Combien de comparaisons faut-il pour trouver le maximum de 1 000 000 de valeurs ?
- 🟡 Combien d’étapes, au plus, la dichotomie demande-t-elle pour un tableau trié d’un milliard d’éléments ?
- 🔴 Mesurez l’exposant : tracez $\log(\text{temps})$ en fonction de $\log(n)$ pour le tri à bulles et estimez la pente par régression linéaire. Retrouvez-vous 2 ?

In [ ]:
mes_comparaisons = None             # 🟢
verifier('Comparaisons pour n = 1 000 000', mes_comparaisons, 10**6 - 1, tol=.5)
mes_etapes = None                   # 🟡 (entier)
verifier('Dichotomie sur 10⁹ éléments', mes_etapes, math.floor(math.log2(10**9)) + 1, tol=.5)

## 5 — Calcul matriciel, CPU/GPU et kernel par blocs  *(diapos 07–09)*

Le calcul typique de l’IA est le produit matriciel $C=AB$, avec $C_{ij}=\sum_k A_{ik}B_{kj}$. Pour des matrices $n\times n$ : $n^3$ multiplications, indépendantes d’une case à l’autre — d’où l’intérêt du **parallélisme**.

Ci-dessous, la même opération écrite de deux façons : trois boucles Python (une multiplication à la fois) et l’opérateur `@` de NumPy, qui délègue à une bibliothèque optimisée. C’est une **analogie** du passage « séquentiel → parallèle », pas une mesure CPU contre GPU.

In [ ]:
def produit_boucles(A, B):
    n, m, p = A.shape[0], A.shape[1], B.shape[1]
    C = np.zeros((n, p))
    for i in range(n):
        for j in range(p):
            s = 0.0
            for k in range(m):
                s += A[i, k] * B[k, j]
            C[i, j] = s
    return C

n = 60
A, B = rng.normal(size=(n, n)), rng.normal(size=(n, n))
t = time.perf_counter(); C1 = produit_boucles(A, B); t1 = time.perf_counter() - t
t = time.perf_counter(); C2 = A @ B; t2 = time.perf_counter() - t
assert np.allclose(C1, C2)
print(f'n = {n} : {n**3:,} multiplications'.replace(',', ' '))
print(f'Boucles Python : {t1 * 1000:.1f} ms | NumPy : {t2 * 1000:.3f} ms | rapport ≈ {t1 / max(t2, 1e-9):.0f}×')

### Un « kernel » qui traite un vecteur par blocs, avec masque

Sur GPU, un **kernel** est une fonction appliquée en parallèle à des blocs d’éléments. Si la taille du vecteur n’est pas un multiple de la taille de bloc, le dernier bloc déborde : un **masque** désactive les positions hors limites. C’est exactement ce que l’on exprime en Triton — simulé ici en NumPy.

> Ne pas confondre le kernel du système d’exploitation et un kernel GPU. Triton n’est pas un modèle d’IA.

In [ ]:
def kernel_addition(a, b, taille_bloc=8):
    n = len(a); c = np.zeros(n); nb_blocs = math.ceil(n / taille_bloc)
    for bloc in range(nb_blocs):                      # sur GPU, ces blocs s'exécutent en parallèle
        indices = bloc * taille_bloc + np.arange(taille_bloc)
        masque = indices < n                          # positions valides
        idx = indices[masque]
        c[idx] = a[idx] + b[idx]
        print(f'bloc {bloc + 1} : indices {indices[0]}–{indices[-1]} | positions masquées : {(~masque).sum()}')
    return c

a, b = np.arange(18.0), 10 * np.arange(18.0)
assert np.allclose(kernel_addition(a, b, 8), a + b)

**À faire**

- 🟢 Pour deux matrices $100\times100$, combien de multiplications ? Et pour $1000\times1000$ ?
- 🟡 Modifiez `n` (30, 60, 120). Par combien le temps des boucles est-il multiplié quand $n$ double ? Est-ce cohérent avec $n^3$ ?
- 🔴 Écrivez `kernel_produit_scalaire(a, b, taille_bloc)` : chaque bloc calcule une somme partielle masquée, puis on additionne les sommes partielles. Vérifiez avec `a @ b`.

In [ ]:
mes_multiplications = None          # 🟢 pour n = 1000
verifier('Multiplications pour n = 1000', mes_multiplications, 1000**3, tol=.5)

## 6 — IA symbolique, statistique et hybride  *(diapos 10–14)*

- **Symbolique :** une règle écrite, explicable : `si note >= 10 : valide`.
- **Statistique :** un modèle $f_\theta$ ajusté sur des exemples ; il s’adapte, mais peut se tromper.
- **Hybride :** le modèle prédit, une contrainte métier vérifie.

Les données ci-dessous sont **synthétiques** : 5 % des étiquettes sont volontairement erronées, comme dans un vrai fichier.

In [ ]:
from sklearn.linear_model import LogisticRegression
g = np.random.default_rng(SEED)
notes_ex = g.uniform(0, 20, 200).round(1)
valide = (notes_ex >= 10).astype(int)
bruit = g.random(200) < .05
valide_observe = np.where(bruit, 1 - valide, valide)          # étiquettes observées, imparfaites

regle = lambda note: int(note >= 10)                                              # symbolique
modele = LogisticRegression().fit(notes_ex.reshape(-1, 1), valide_observe)         # statistique
frontiere = -modele.intercept_[0] / modele.coef_[0, 0]
print(f'Frontière apprise : note ≈ {frontiere:.2f} (la règle écrite dit 10)')

def hybride(note):
    proposition = int(modele.predict([[note]])[0])
    return proposition if proposition == regle(note) else regle(note)            # la contrainte métier tranche

for note in [8.0, 9.8, 10.2, 13.0]:
    p = modele.predict_proba([[note]])[0, 1]
    print(f'note {note:>4} | règle : {regle(note)} | modèle : {int(p >= .5)} (p = {p:.2f}) | hybride : {hybride(note)}')

**À faire**

- 🟢 Donnez un avantage et une limite de la règle écrite, puis du modèle appris.
- 🟡 Passez le taux d’étiquettes erronées de 5 % à 30 %. Que devient la frontière apprise ? Pourquoi ?
- 🔴 Dans quel cas l’approche hybride serait-elle **moins** bonne que le modèle seul ? Construisez un exemple où la règle écrite est elle-même fausse.

## 7 — Apprendre, c’est minimiser ; trois paradigmes  *(diapos 15–16)*

$$\theta^*=\arg\min_\theta\frac1n\sum_{i=1}^n\ell\big(f_\theta(x_i),y_i\big).$$

Exemple minimal : $f_\theta(x)=\theta x$ et perte quadratique. Le minimum se calcule : $\theta^*=\dfrac{\sum x_iy_i}{\sum x_i^2}$.

In [ ]:
x = np.array([1., 2., 3., 4.]); y = np.array([2.1, 3.9, 6.2, 7.8])
perte = lambda theta: np.mean((theta * x - y) ** 2)
thetas = np.linspace(0, 4, 200)
theta_opt = (x * y).sum() / (x * x).sum()
plt.plot(thetas, [perte(t) for t in thetas]); plt.axvline(theta_opt, ls='--', color='green')
plt.xlabel('θ'); plt.ylabel('perte moyenne'); plt.title(f'Minimum en θ* = {theta_opt:.2f}'); plt.show()

### Trois paradigmes en trois mini-expériences

| Paradigme | Données | Ci-dessous |
|---|---|---|
| Supervisé | entrées $X$ **et** cibles $y$ | régression sur les points précédents |
| Non supervisé | entrées $X$ seulement | regroupement de points en 2 groupes |
| Renforcement | états, actions, **récompenses** | un agent choisit entre 3 machines à sous |

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
# 1) Supervisé
reg = LinearRegression().fit(x.reshape(-1, 1), y)
print(f'Supervisé      : ŷ = {reg.coef_[0]:.2f}·x + {reg.intercept_:.2f} → prédiction pour x = 5 : {reg.predict([[5]])[0]:.2f}')
# 2) Non supervisé
g = np.random.default_rng(SEED)
points = np.vstack([g.normal([0, 0], .6, (30, 2)), g.normal([4, 4], .6, (30, 2))])
groupes = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit_predict(points)
print('Non supervisé  : tailles des groupes trouvés =', np.bincount(groupes))
# 3) Renforcement : bandit à 3 bras, stratégie ε-gloutonne
gains_reels = [.2, .5, .8]                 # inconnus de l'agent
estime, essais, total, eps = np.zeros(3), np.zeros(3), 0, .1
for t in range(2000):
    bras = g.integers(3) if g.random() < eps else int(estime.argmax())
    recompense = float(g.random() < gains_reels[bras])
    essais[bras] += 1; estime[bras] += (recompense - estime[bras]) / essais[bras]; total += recompense
print('Renforcement   : gains estimés =', estime.round(2), '| essais par bras =', essais.astype(int), '| gain moyen =', round(total / 2000, 3))

**À faire**

- 🟢 Pour chacune des trois expériences, dites ce qui joue le rôle du « signal d’apprentissage ».
- 🟡 Calculez $\theta^*$ à la main avec la formule, puis vérifiez.
- 🔴 Dans le bandit, testez $\varepsilon=0$, $0{,}1$ et $0{,}5$ sur 20 graines. Tracez le gain moyen et expliquez le compromis exploration / exploitation.

In [ ]:
mon_theta = None                    # 🟡
verifier('θ* = Σxy / Σx²', mon_theta, theta_opt)

## 8 — Un modèle génératif en vingt lignes  *(diapos 18–19)*

Un modèle de langage prédit le **symbole suivant** à partir du contexte. Version la plus simple : un **modèle de bigrammes de caractères** — on compte, dans un corpus, quel caractère suit quel autre, puis on tire au hasard selon ces fréquences.

Ce modèle produit des suites de lettres qui *ressemblent* à du français sans rien comprendre. Un LLM fait la même chose à une tout autre échelle, avec un contexte long et des milliards de paramètres : **fluide ne veut pas dire exact**.

In [ ]:
corpus = ("l'intelligence artificielle apprend a partir de donnees. un modele predit le mot suivant. "
          "la machine calcule mais ne comprend pas. verifier les sources reste indispensable. "
          "les donnees et les algorithmes forment la base de l'apprentissage automatique. "
          "le modele generatif produit un texte plausible sans garantir sa veracite. ")
alphabet = sorted(set(corpus)); indice = {c: i for i, c in enumerate(alphabet)}
comptes = np.ones((len(alphabet), len(alphabet))) * .01           # lissage
for c1, c2 in zip(corpus, corpus[1:]):
    comptes[indice[c1], indice[c2]] += 1

def generer(debut='l', longueur=120, temperature=1.0, graine=SEED):
    g = np.random.default_rng(graine); sortie = debut
    for _ in range(longueur):
        logits = np.log(comptes[indice[sortie[-1]]]) / temperature
        p = np.exp(logits - logits.max()); p /= p.sum()
        sortie += alphabet[g.choice(len(alphabet), p=p)]
    return sortie

for T in [0.3, 1.0, 2.0]:
    print(f'T = {T} → {generer(temperature=T)}\n')

**À faire**

- 🟢 Changez la lettre de départ et la température. Que se passe-t-il quand la température est très basse ? très haute ?
- 🟡 Affichez les 5 caractères les plus probables après la lettre `e`. Est-ce cohérent avec le corpus ?
- 🔴 Passez à un modèle de **trigrammes** (contexte de 2 caractères). Le texte est-il meilleur ? Que se passe-t-il si le corpus est très petit ? Reliez votre réponse à la mémorisation et au surapprentissage.

## 9 — Atelier de la conférence : défi en binôme (20 minutes)  *(diapo 20)*

À partir d’un petit tableau de notes :

1. identifier les données, la cible et les traitements ;
2. écrire le calcul de moyenne en Python ;
3. comparer ce programme classique à un classificateur entraîné sur des exemples ;
4. définir une évaluation sur des données jamais vues.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
g = np.random.default_rng(7)
tableau = pd.DataFrame(g.integers(3, 20, size=(60, 3)), columns=['Maths', 'Info', 'Anglais'])
tableau['Moyenne'] = tableau[['Maths', 'Info', 'Anglais']].mean(axis=1).round(2)       # programme classique
tableau['Valide'] = (tableau['Moyenne'] >= 10).astype(int)                             # règle écrite
display(tableau.head(8))

# Classificateur : il ne connaît PAS la règle, seulement des exemples
X_tr, X_te, y_tr, y_te = train_test_split(tableau[['Moyenne']], tableau['Valide'], test_size=.3, random_state=SEED)
arbre = DecisionTreeClassifier(max_depth=1, random_state=SEED).fit(X_tr, y_tr)
print(f'Seuil retrouvé par apprentissage : moyenne ≈ {arbre.tree_.threshold[0]:.2f} (la règle écrite dit 10)')
print(f'Exactitude sur {len(X_te)} étudiants jamais vus : {arbre.score(X_te, y_te):.2f}')

**Questions du défi**

- 🟢 Pourquoi le seuil appris n’est-il pas exactement 10 ?
- 🟡 Réentraînez avec seulement 10 exemples. Le seuil est-il plus ou moins précis ?
- 🔴 Remplacez la cible par une « réussite » bruitée (10 % d’erreurs). Quels risques y aurait-il à utiliser ce classificateur pour décider du sort d’un étudiant réel ?

## Mini-projets (à rendre)

| Niveau | Sujet | Livrable |
|---|---|---|
| 🟢 L1–L2 | **Carte d’identité d’un fichier** : choisir un texte, calculer sa taille en bits, son entropie par caractère et ses 10 symboles les plus fréquents. | Notebook + 5 lignes de commentaire |
| 🟡 L3 | **Canal bruité** : comparer les codes à répétition ×3 et ×5 pour $p$ de 0,01 à 0,4 (simulation et théorie sur le même graphique). | Notebook + 1 figure légendée |
| 🔴 M1–M2 | **Mini-modèle de langage** : $n$-grammes de caractères pour $n=1,2,3,4$ ; mesurer la perte moyenne $-\log_2 p$ sur un texte **non vu** et discuter le surapprentissage. | Notebook + rapport d’une page |

## Barème (100 points)

| Critère | Points |
|---|---:|
| Exécution sans erreur, reproductibilité (graine fixée) | 15 |
| Exercices du niveau choisi, calculs à la main justifiés | 35 |
| Clarté du code et des figures (axes, légendes, unités) | 20 |
| Interprétation et esprit critique | 20 |
| Mini-projet | 10 |

## Autoévaluation

1. Quelle différence entre une donnée, une information et une connaissance ?
2. Que vaut l’entropie d’une pièce équilibrée ? d’une pièce truquée à 100 % ?
3. Pourquoi une image de $100\times100$ pixels à 8 bits occupe-t-elle 80 000 bits ?
4. Pourquoi deux algorithmes corrects peuvent-ils être l’un utilisable, l’autre non ?
5. Pourquoi le produit matriciel se prête-t-il au parallélisme ?
6. Citez un avantage et une limite d’une règle symbolique et d’un modèle appris.
7. Qu’est-ce qui distingue les trois paradigmes d’apprentissage ?
8. Pourquoi un texte fluide n’est-il pas nécessairement exact ?

**Suite :** TP 2 — apprentissage supervisé.